# Module 01 · Unit 02
# Truth Tables

| | |
|---|---|
| **Estimated time** | 55–65 min |
| **Exercises** | Download PDF from the course repository |
| **Security thread** | Access Control & Policy Logic \[AC\] |
| **Refresher** | Module 01 · Unit 01 — Propositions and Logical Connectives |

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Construct a truth table systematically for any formula with $n$ variables
2. Classify a formula as a **tautology**, **contradiction**, or **satisfiable**
3. Prove logical equivalence by truth table
4. Formally verify De Morgan's Laws and the conditional identity $P \rightarrow Q \equiv \neg P \vee Q$
5. Explain what a **missing check vulnerability** is in an access control policy
6. Use Python to detect which policy inputs are incorrectly granted access by a buggy implementation


## 🔣 Symbol Primer

Two new symbols appear in this notebook. All others carry over from Units 00–01.

| Symbol | Name | Read it as | Meaning |
|---|---|---|---|
| $\top$ | Tautology / Verum | "top" / "always true" | A formula that is true for **every** input combination |
| $\bot$ | Contradiction / Falsum | "bottom" / "always false" | A formula that is false for **every** input combination |
| $\equiv$ | Logical equivalence | "is equivalent to" | Two formulas with **identical** truth table columns *(from Unit 01)* |
| $\neg, \wedge, \vee, \rightarrow, \leftrightarrow$ | Connectives | *(see Unit 01 primer)* | Not, And, Or, If-then, Iff |

> **Quick orientation:** $\top$ and $\bot$ are the two extremes.
> A tautology ($\top$) is true no matter what — it carries no information.
> A contradiction ($\bot$) is false no matter what — it is an impossible claim.
> Everything interesting lives in between: **satisfiable** formulas that are true
> for some inputs and false for others.

---


## 1 · The Truth Table Method

A **truth table** for a formula with $n$ variables has $2^n$ rows — one for each
possible assignment of truth values to those variables. We enumerate rows in a
fixed order: the rightmost variable alternates T/F every row; each variable to its
left alternates half as fast.

For three variables $A$, $T$, $S$:

| Row | $A$ | $T$ | $S$ |
|:---:|:---:|:---:|:---:|
| 1 | T | T | T |
| 2 | T | T | F |
| 3 | T | F | T |
| 4 | T | F | F |
| 5 | F | T | T |
| 6 | F | T | F |
| 7 | F | F | T |
| 8 | F | F | F |

This is binary counting in disguise: T = 1, F = 0, reading each row as a 3-bit number
from 7 down to 0.

### Evaluating a Compound Formula

To evaluate a complex formula, work from the **inside out** — innermost parentheses
and tightest-binding connectives first, building intermediate columns:

**Example:** Evaluate $A \wedge T \wedge \neg S$ column by column.

| $A$ | $T$ | $S$ | $\neg S$ | $A \wedge T$ | $A \wedge T \wedge \neg S$ |
|:---:|:---:|:---:|:---:|:---:|:---:|
| T | T | T | F | T | **F** |
| T | T | F | T | T | **T** |
| T | F | T | F | F | **F** |
| T | F | F | T | F | **F** |
| F | T | T | F | F | **F** |
| F | T | F | T | F | **F** |
| F | F | T | F | F | **F** |
| F | F | F | T | F | **F** |

Only **one row** grants access: the admin user, within the time window, and not suspended.
This is the principle of least privilege expressed as a truth table.


## 2 · Tautologies, Contradictions, and Satisfiability

### Tautology — $\top$

A formula is a **tautology** if its final column is all T — it is true under every
possible assignment.

$$P \vee \neg P \;\equiv\; \top \qquad \text{(Law of Excluded Middle)}$$

| $P$ | $\neg P$ | $P \vee \neg P$ |
|:---:|:---:|:---:|
| T | F | **T** |
| F | T | **T** |

**Security relevance.** A policy that is a tautology grants access unconditionally —
it is effectively no policy at all. If through algebraic simplification or a bug
a compound access rule reduces to $\top$, every user is permitted. This is the
logic-level equivalent of a firewall rule that matches all traffic.

### Contradiction — $\bot$

A formula is a **contradiction** if its final column is all F — it can never be true.

$$P \wedge \neg P \;\equiv\; \bot$$

| $P$ | $\neg P$ | $P \wedge \neg P$ |
|:---:|:---:|:---:|
| T | F | **F** |
| F | T | **F** |

**Security relevance.** A policy that is a contradiction denies access to everyone
— an availability failure. This can happen when two conflicting constraints are
ANDed together by mistake: "must be admin AND must not be admin."

### Satisfiable

A formula is **satisfiable** if at least one row of its truth table is T — there
exists some input that makes it true. Tautologies are trivially satisfiable. The
interesting class is formulas that are satisfiable but not tautologies: they grant
access to some users and deny others. Every well-formed access policy belongs here.

$$\text{tautology} \;\Rightarrow\; \text{satisfiable} \qquad \text{(but not conversely)}$$


## 3 · Proving Logical Equivalence by Truth Table

Two formulas $\phi$ and $\psi$ are **logically equivalent**, written
$\phi \equiv \psi$, if and only if they produce identical output columns in
their combined truth table — that is, $\phi \leftrightarrow \psi$ is a tautology:

$$\phi \equiv \psi \;\Longleftrightarrow\; (\phi \leftrightarrow \psi) \equiv \top$$

**Method:** add a column for each formula, then verify the columns are identical row
by row. If any row differs, the formulas are not equivalent — that row is a
counterexample (connecting back to Module 00).

### Why This Matters for Security

Logical equivalence is the foundation of policy refactoring. If you can show that a
simpler expression is equivalent to a complex policy, you can replace one with the
other without changing who gets access. It is also how you prove that a
re-implementation is faithful to the original specification.

In Unit 03 we will use equivalence to simplify ABAC policies with Boolean algebra
laws. The truth table method provides the ground truth that algebraic simplification
must match.


## 4 · De Morgan's Laws and the Conditional Identity

### De Morgan's Laws — Formal Proof

We stated these in Unit 01 and promised a truth table proof. Here it is.

**Law 1:** $\neg(P \wedge Q) \;\equiv\; (\neg P \vee \neg Q)$

| $P$ | $Q$ | $P \wedge Q$ | $\neg(P \wedge Q)$ | $\neg P$ | $\neg Q$ | $\neg P \vee \neg Q$ | Match? |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| T | T | T | **F** | F | F | **F** | ✓ |
| T | F | F | **T** | F | T | **T** | ✓ |
| F | T | F | **T** | T | F | **T** | ✓ |
| F | F | F | **T** | T | T | **T** | ✓ |

Every row matches. The columns are identical. $\neg(P \wedge Q) \equiv \neg P \vee \neg Q$. $\square$

**Law 2:** $\neg(P \vee Q) \;\equiv\; (\neg P \wedge \neg Q)$

The proof is symmetric — we verify it with Python below.

### The Conditional Identity

The conditional is secretly a disjunction:

$$P \rightarrow Q \;\equiv\; \neg P \vee Q$$

This is why the conditional has three True rows — it shares its structure with $\vee$,
with $P$ negated on the left. This identity is the key to simplifying policy
obligations algebraically, and to negating them correctly:

$$\neg(P \rightarrow Q) \;\equiv\; \neg(\neg P \vee Q) \;\equiv\; P \wedge \neg Q$$

In English: the **violation** of "if privileged then logged" is exactly
"privileged AND not logged" — which is precisely what a security audit looks for.


## 5 · The Missing Check Vulnerability

We now have all the tools to formally analyse a real class of access control bug.

### The Policy

Recall the ABAC policy from Unit 01:

$$\text{allow} \;\equiv\; A \wedge T \wedge \neg S$$

where $A$ = admin, $T$ = time in window, $S$ = suspended.

### The Bug

Suppose a developer implements the policy in a system where the suspension flag $S$
is read from a user session object. Due to a bug in the session initialisation code,
$S$ is never populated — it silently defaults to $\mathtt{False}$ (not suspended)
for every user.

The effective policy becomes:

$$\text{allow\_buggy} \;\equiv\; A \wedge T \wedge \neg\,\mathtt{False} \;\equiv\; A \wedge T \wedge \mathtt{True} \;\equiv\; A \wedge T$$

The suspension check has vanished. Its negation $\neg S = \neg\,\mathtt{False} = \mathtt{True}$
is now a tautology fragment — it contributes nothing and can be dropped.

### The Consequence

Using De Morgan's Law 1, the **set of users the bug exposes** is:

$$\text{allow\_buggy} \wedge \neg\,\text{allow\_correct}$$

$$= (A \wedge T) \wedge \neg(A \wedge T \wedge \neg S)$$

$$= (A \wedge T) \wedge (\neg A \vee \neg T \vee S)$$

Since $A \wedge T$ is true, $\neg A$ and $\neg T$ are both false, leaving:

$$= A \wedge T \wedge S$$

**Exactly the suspended admins within the time window.** These are the users that
the correct policy explicitly blocks but the buggy policy admits. The truth table
makes this precise — we can enumerate them exactly.


---

## 🔐 Security Bridge &nbsp;·&nbsp; \[AC\]

| Concept | Security translation |
|---|---|
| **Tautology** $\top$ | Policy that admits everyone — broken access control |
| **Contradiction** $\bot$ | Policy that admits nobody — availability failure |
| **Satisfiable** | Well-formed policy: some users in, some out |
| **Logical equivalence** | Policy refactoring without changing access decisions |
| **De Morgan's Laws** | Convert allow-conditions to deny-conditions and back |
| **Conditional identity** | Expose the violation condition of any obligation rule |
| **Missing check** | A conjunct that defaults to $\top$, silently widening the gate |

**The pattern to internalise.** Every access control vulnerability of the
"missing check" class has the same shape in logic:

1. The correct policy is $\phi_1 \wedge \phi_2 \wedge \cdots \wedge \phi_k$.
2. A bug causes one clause $\phi_i$ to evaluate as $\top$.
3. The effective policy becomes $\phi_1 \wedge \cdots \wedge \top \wedge \cdots \wedge \phi_k$,
   which simplifies to $\phi_1 \wedge \cdots \wedge \phi_k$ *without* $\phi_i$.
4. The exposed users are exactly those who satisfy all remaining clauses but
   **fail** $\phi_i$ — the set $A \wedge T \wedge \cdots \wedge \neg\phi_i$.

This algebraic derivation is how a security engineer can reason about the *blast
radius* of a missing check before seeing a single user log.

---


## Python: Visualization & Verification

**1 · ABAC Truth Table** — generate and visualise the full 8-row truth table for
$A \wedge T \wedge \neg S$, building intermediate columns step by step.

**2 · Tautology and Contradiction Detector** — classify any formula automatically
and visualise the satisfying fraction across a set of example formulas.

**3 · Equivalence Verifier** — formally verify De Morgan's Laws and the conditional
identity by comparing truth table columns and flagging any mismatch.

**4 · Missing Check Vulnerability** — compare the correct and buggy ABAC policies
row by row, highlight the exposed users, and confirm the algebraic derivation above.


In [ ]:
import subprocess, sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

for pkg in ["numpy", "matplotlib", "sympy", "scipy", "networkx"]:
    install(pkg)

print("All packages installed.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import itertools

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (9, 6), 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'lines.linewidth': 2, 'figure.dpi': 120,
})

TS_BLUE  = '#1e64b4'
TS_AMBER = '#c87814'
TS_GREEN = '#1e8c50'
TS_GRAY  = '#64646e'
TS_RED   = '#b41e1e'
TS_LIGHT = '#f5f7fa'

def tf(b):
    """Boolean to T/F string."""
    return 'T' if b else 'F'

print('Setup complete.')


### 1 · ABAC Policy Truth Table — Built Step by Step

We construct the truth table for $A \wedge T \wedge \neg S$ column by column,
exactly as Section 1 described. The intermediate columns $\neg S$ and $A \wedge T$
are shown explicitly so the evaluation path is transparent.

The final column is colour-coded: green = access granted, red = access denied.


In [ ]:
# ── 1 · ABAC Policy Truth Table ───────────────────────────────────────────────

combos = list(itertools.product([True, False], repeat=3))  # 2³ = 8 rows

# Build all columns
rows = []
for A, T, S in combos:
    not_S   = not S
    A_and_T = A and T
    allow   = A and T and not S
    rows.append((A, T, S, not_S, A_and_T, allow))

# ── Visual table ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.axis('off')

col_headers = ['A\n(admin)', 'T\n(in window)', 'S\n(suspended)',
               '¬S', 'A ∧ T', 'A ∧ T ∧ ¬S\n(allow?)']

cell_text   = [[tf(v) for v in row] for row in rows]
cell_colors = []

for row in rows:
    A, T, S, not_S, A_and_T, allow = row
    base = ['#f8f9fa'] * 3
    base.append(TS_GREEN if not_S   else TS_RED)
    base.append(TS_GREEN if A_and_T else TS_RED)
    base.append(TS_GREEN if allow   else TS_RED)
    cell_colors.append(base)

tbl = ax.table(
    cellText=cell_text,
    colLabels=col_headers,
    cellColours=cell_colors,
    cellLoc='center',
    loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1.2, 1.8)

for j in range(len(col_headers)):
    tbl[(0, j)].set_facecolor(TS_BLUE)
    tbl[(0, j)].set_text_props(color='white', fontweight='bold')

ax.set_title(
    'ABAC Policy: $A \\wedge T \\wedge \\neg S$  —  '
    'Step-by-step truth table ($2^3 = 8$ rows)',
    pad=14, fontsize=11
)
plt.tight_layout()
plt.show()

allow_count = sum(r[-1] for r in rows)
print(f"Rows granting access:  {allow_count} / {len(rows)}")
print(f"Rows denying access:   {len(rows) - allow_count} / {len(rows)}")
print(f"\nThe one green row: A=T, T=T, S=F  →  admin, in-window, not suspended.")


**What do you see?**

Only **one** of eight possible input combinations grants access — row 2:
admin user ($A = T$), within the time window ($T = T$), and not suspended ($S = F$).
The principle of least privilege expressed as a truth table: the policy is as
restrictive as logically possible while still admitting any user at all.

Notice the intermediate columns:
- $\neg S$ flips the suspension flag — this is the check the bug will later remove.
- $A \wedge T$ grants two green cells — but the final $\wedge \neg S$ kills one of them
  (row 1, where $S = T$).

**Try this:** Change the policy to $(A \vee Q) \wedge T \wedge \neg S$ where
$Q$ = "is resource owner." How many rows now grant access? Is the policy still
satisfiable? Is it more or less restrictive?


### 2 · Tautology and Contradiction Detector

The function below classifies any formula as tautology ($\top$), contradiction
($\bot$), or satisfiable, and returns the fraction of inputs that satisfy it.
We apply it to five formulas: two corner cases and three policy-shaped examples.


In [ ]:
# ── 2 · Tautology / Contradiction / Satisfiability Classifier ────────────────

def classify(variables, formula):
    """Return (classification, satisfy_count, total_rows)."""
    combos   = list(itertools.product([True, False], repeat=len(variables)))
    results  = [formula(**dict(zip(variables, c))) for c in combos]
    sat      = sum(results)
    total    = len(results)
    if sat == total:
        label = 'Tautology  (⊤)'
    elif sat == 0:
        label = 'Contradiction  (⊥)'
    else:
        label = 'Satisfiable'
    return label, sat, total

formulas = [
    ('$P \\vee \\neg P$',
     ['P'], lambda P: P or not P),
    ('$P \\wedge \\neg P$',
     ['P'], lambda P: P and not P),
    ('$A \\wedge T \\wedge \\neg S$\n(ABAC policy)',
     ['A','T','S'], lambda A,T,S: A and T and not S),
    ('$(A \\wedge T) \\vee \\neg T$\n(over-permissive)',
     ['A','T'], lambda A,T: (A and T) or not T),
    ('$A \\wedge \\neg A$\n(conflicting rule)',
     ['A'], lambda A: A and not A),
]

labels, sat_fracs, classifications = [], [], []
for name, variables, formula in formulas:
    cls, sat, total = classify(variables, formula)
    labels.append(name)
    sat_fracs.append(sat / total)
    classifications.append(cls)
    print(f"{name.replace(chr(10),' '):<40}  {cls:<28}  {sat}/{total} rows satisfy")

print()

# ── Bar chart ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 4.5))

bar_colors = []
for cls in classifications:
    if 'Tautology'     in cls: bar_colors.append(TS_AMBER)
    elif 'Contradiction' in cls: bar_colors.append(TS_RED)
    else:                        bar_colors.append(TS_BLUE)

bars = ax.barh(range(len(labels)), sat_fracs, color=bar_colors,
               edgecolor='white', height=0.55)

for i, (frac, cls) in enumerate(zip(sat_fracs, classifications)):
    ax.text(frac + 0.02, i, cls, va='center', fontsize=9, color=TS_GRAY)

ax.set_xlim(0, 1.55)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('Fraction of inputs that satisfy the formula')
ax.set_title('Formula Classification: Tautology · Satisfiable · Contradiction', pad=10)
ax.axvline(1.0, color=TS_GRAY, linestyle='--', lw=1, alpha=0.5)

legend_handles = [
    mpatches.Patch(color=TS_AMBER, label='Tautology ⊤ — always true'),
    mpatches.Patch(color=TS_BLUE,  label='Satisfiable — sometimes true'),
    mpatches.Patch(color=TS_RED,   label='Contradiction ⊥ — never true'),
]
ax.legend(handles=legend_handles, loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()


**What do you see?**

- $P \vee \neg P$ reaches the full bar — every input satisfies it. As a policy,
  it is useless: it admits everyone unconditionally.
- $P \wedge \neg P$ and $A \wedge \neg A$ have zero bar — they are impossible
  conditions that block everyone. A policy accidentally containing such a clause
  would cause a silent denial-of-service.
- The ABAC policy sits at $1/8$ — only one of eight combinations grants access.
  This is precisely the level of restrictiveness we designed.
- The over-permissive rule $(A \wedge T) \vee \neg T$ satisfies $3/4$ of inputs.
  The $\vee \neg T$ branch admits anyone outside the time window — the opposite
  of the intended restriction. This is a classic OR misconfiguration.

**The security principle:** a well-formed access policy should be satisfiable but far
from a tautology. If the satisfy-fraction approaches 1 after a policy change,
that is a signal to audit the change carefully.


### 3 · Equivalence Verifier — De Morgan's Laws and the Conditional Identity

We now formally verify three of the most important equivalences in logic. For each
pair, we generate both truth table columns and confirm they are identical for every
input. A mismatch on any row would be a counterexample — proof the equivalence fails.


In [ ]:
# ── 3 · Equivalence Verifier ─────────────────────────────────────────────────

def verify_equivalence(variables, phi, phi_label, psi, psi_label):
    """Verify phi ≡ psi by comparing truth table columns. Returns True if equivalent."""
    combos  = list(itertools.product([True, False], repeat=len(variables)))
    phi_col = [phi(**dict(zip(variables, c))) for c in combos]
    psi_col = [psi(**dict(zip(variables, c))) for c in combos]
    match   = [a == b for a, b in zip(phi_col, psi_col)]
    return phi_col, psi_col, match

equivalences = [
    (['P','Q'],
     lambda P,Q: not (P and Q),         '¬(P ∧ Q)',
     lambda P,Q: (not P) or (not Q),    '¬P ∨ ¬Q',
     "De Morgan's Law 1"),
    (['P','Q'],
     lambda P,Q: not (P or Q),          '¬(P ∨ Q)',
     lambda P,Q: (not P) and (not Q),   '¬P ∧ ¬Q',
     "De Morgan's Law 2"),
    (['P','Q'],
     lambda P,Q: (not P) or Q,          'P → Q',
     lambda P,Q: (not P) or Q,          '¬P ∨ Q',
     'Conditional Identity'),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, (variables, phi, phi_lbl, psi, psi_lbl, title) in zip(axes, equivalences):
    phi_col, psi_col, match = verify_equivalence(variables, phi, phi_lbl, psi, psi_lbl)
    n = len(phi_col)
    combos = list(itertools.product([True, False], repeat=len(variables)))

    # Build display matrix: col0=phi, col1=psi, col2=match
    matrix = np.array([
        [float(v) for v in phi_col],
        [float(v) for v in psi_col],
        [float(v) for v in match],
    ]).T  # shape: (n_rows, 3)

    cmap_main  = ListedColormap([TS_RED, TS_GREEN])
    cmap_match = ListedColormap(['#b41e1e', '#1e8c50'])

    ax.imshow(matrix, cmap=cmap_main, vmin=0, vmax=1, aspect='auto')

    row_labels = [', '.join(tf(v) for v in c) for c in combos]
    for r in range(n):
        for col_i in range(3):
            if col_i == 2:
                txt = '✓' if match[r] else '✗'
                color = 'white'
            else:
                txt   = tf(bool(matrix[r, col_i]))
                color = 'white'
            ax.text(col_i, r, txt, ha='center', va='center',
                    fontsize=11, fontweight='bold', color=color)

    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels([phi_lbl, psi_lbl, 'Match?'], fontsize=8)
    ax.set_yticks(range(n))
    ax.set_yticklabels(row_labels, fontsize=8)
    ax.set_title(title, fontsize=10, fontweight='bold', pad=8)
    ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)

    all_match = all(match)
    status = '≡  EQUIVALENT ✓' if all_match else '≢  NOT EQUIVALENT ✗'
    ax.set_xlabel(status, fontsize=9,
                  color=TS_GREEN if all_match else TS_RED, fontweight='bold')

plt.suptitle("Formal Verification by Truth Table", fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**What do you see?**

All three pairs show identical columns with ✓ in every match cell — each equivalence
is formally verified. Notice:

- **De Morgan's Law 1:** $\neg(P \wedge Q)$ and $\neg P \vee \neg Q$ are green in
  exactly the same rows. A counterexample (any single ✗) would disprove the law.
- **De Morgan's Law 2:** the symmetric result — negation distributes over $\vee$
  and flips it to $\wedge$.
- **Conditional Identity:** $P \rightarrow Q$ and $\neg P \vee Q$ are identical by
  definition — the conditional *is* the disjunction with a negated left side.

**The verification pattern matters beyond this notebook.** Whenever you refactor an
access policy expression — pulling out a common clause, applying De Morgan, simplifying
a double negation — the truth table is the ground-truth check. If the refactored
formula produces even one different row, the refactoring is wrong.


### 4 · The Missing Check Vulnerability — Full Analysis

We now run the full truth table analysis from Section 5. The correct policy is
$A \wedge T \wedge \neg S$. The buggy policy — where $S$ defaults to $\mathtt{False}$
— effectively becomes $A \wedge T$.

We compare them row by row and highlight the **exposed rows**: cases where the buggy
policy grants access but the correct policy does not. Our algebraic derivation
predicted these would be exactly the rows where $A = T$, $T = T$, $S = T$ — suspended
admins within the time window.


In [ ]:
# ── 4 · Missing Check Vulnerability ──────────────────────────────────────────

combos = list(itertools.product([True, False], repeat=3))

correct = lambda A,T,S: A and T and not S   # A ∧ T ∧ ¬S
buggy   = lambda A,T,S: A and T             # A ∧ T   (¬S silently True)

results = []
for A, T, S in combos:
    c = correct(A=A, T=T, S=S)
    b = buggy(A=A, T=T, S=S)
    exposed = b and not c   # buggy grants, correct denies
    results.append((A, T, S, c, b, exposed))

# ── Side-by-side table ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
ax.axis('off')

col_headers = ['A\n(admin)', 'T\n(in window)', 'S\n(suspended)',
               'Correct policy\nA∧T∧¬S', 'Buggy policy\nA∧T', 'Exposed?']

cell_text   = [[tf(v) for v in row] for row in results]
cell_colors = []

for A, T, S, c, b, exposed in results:
    row_c = ['#f8f9fa'] * 3
    row_c.append(TS_GREEN if c else TS_RED)
    row_c.append(TS_GREEN if b else TS_RED)
    row_c.append('#c0392b' if exposed else '#f8f9fa')  # dark red if exposed
    cell_colors.append(row_c)

tbl = ax.table(
    cellText=cell_text,
    colLabels=col_headers,
    cellColours=cell_colors,
    cellLoc='center',
    loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1.2, 1.8)

for j in range(len(col_headers)):
    tbl[(0, j)].set_facecolor(TS_BLUE)
    tbl[(0, j)].set_text_props(color='white', fontweight='bold')

# Highlight exposed rows with bold text
for i, (A, T, S, c, b, exposed) in enumerate(results):
    if exposed:
        tbl[(i+1, 5)].set_text_props(color='white', fontweight='bold')
        tbl[(i+1, 5)].get_text().set_text('⚠ EXPOSED')

ax.set_title(
    'Missing Check: correct policy vs buggy policy\n'
    'Exposed rows = users admitted by the bug but blocked by the correct policy',
    pad=14, fontsize=11
)
plt.tight_layout()
plt.show()

# ── Confirm algebraic prediction ──────────────────────────────────────────────
print("Algebraic prediction: exposed users satisfy A=T, T=T, S=T")
print()
exposed_rows = [(A,T,S) for A,T,S,c,b,exp in results if exp]
for A, T, S in exposed_rows:
    print(f"  A={tf(A)}, T={tf(T)}, S={tf(S)}  →  "
          f"suspended admin in-window (correct=DENY, buggy=ALLOW)")

print()
print(f"Exposed rows: {len(exposed_rows)} / {len(results)}")
print("Prediction confirmed: exactly the suspended admins within the time window.")
print()
print("Security implication:")
print("  Every user with admin role + active session + suspended account")
print("  gets through the gate. The suspension check was the only clause")
print("  blocking them — and it was silently dropped.")


**What do you see?**

Exactly one row is exposed — $A = T$, $T = T$, $S = T$: the suspended admin, within
the time window. The algebraic derivation from Section 5 was correct.

This is the power of the truth table as a security tool. Before writing any code,
before spinning up a test environment, before finding a real suspended admin account
to test with, you can determine *exactly* which users a missing check exposes — by
working through $2^n$ rows with pencil and paper, or with the eight lines of Python
above.

**The broader lesson.** The bug here is silent — no exception is raised, no log
entry is written, the system behaves normally for all non-suspended users. The
only way to detect it is to reason formally about what the policy *should* do versus
what it *does* do. That formal reasoning is exactly what this module is building.

In Unit 03 we will use Boolean algebra to *simplify* and *minimise* access control
policies, and apply these same tools to the full ABAC example with four variables.


In [ ]:
# ── Extension Challenge ───────────────────────────────────────────────────────
#
# 1. BLAST RADIUS
#    Generalise the missing check analysis. Write a function:
#      blast_radius(variables, correct_policy, buggy_policy)
#    that returns the list of input combinations exposed by the bug.
#    Apply it to a four-variable policy: A ∧ T ∧ ¬S ∧ M
#    where M = "MFA verified". What is the blast radius if M is missing?
#
# 2. DOUBLE NEGATION
#    Verify the double negation law:  ¬(¬P) ≡ P
#    Use verify_equivalence. Then find a three-variable policy where a developer
#    accidentally wrote ¬(¬S) instead of S. Does it change access decisions?
#
# 3. TAUTOLOGY TRAP
#    Show algebraically (and then verify with a truth table) that the policy
#      (A → allow) ∧ (¬A → allow)
#    is a tautology. Why is this dangerous? What real-world pattern does it
#    represent? (Hint: think about "default allow" vs "default deny".)

# Your work here:


---

## Summary

| Concept | Definition | Security meaning |
|---|---|---|
| **Tautology** $\top$ | True for all inputs | Policy admits everyone — broken |
| **Contradiction** $\bot$ | False for all inputs | Policy blocks everyone — broken |
| **Satisfiable** | True for some inputs | Well-formed policy |
| **Logical equivalence** $\equiv$ | Identical truth table columns | Safe to refactor one into the other |
| **De Morgan's Law 1** | $\neg(P \wedge Q) \equiv \neg P \vee \neg Q$ | Negate AND → OR the negations |
| **De Morgan's Law 2** | $\neg(P \vee Q) \equiv \neg P \wedge \neg Q$ | Negate OR → AND the negations |
| **Conditional identity** | $P \rightarrow Q \equiv \neg P \vee Q$ | Obligation = negated-premise OR conclusion |
| **Missing check** | A conjunct defaults to $\top$ | Blast radius = rows failing that conjunct |

**The truth table is the ground truth.** Any claim about equivalence, about who gets
access, about the effect of a change — all of it can be verified or refuted by
exhaustive enumeration. For three variables that is 8 rows. For four it is 16.
For ten it is 1,024 — still tractable for a computer, and the *structure* is
always the same.

---

## Up Next

**Module 01 · Unit 03 — Boolean Algebra and Access Control**

We move from exhaustive verification to *algebraic simplification*. Boolean algebra
gives us laws for rewriting policy expressions into equivalent but simpler forms —
without generating a single truth table row. We apply the full toolkit to a
four-variable ABAC policy, minimise it, and prove the minimised form is correct.

$\rightarrow$ `modules/module-01/unit-03-boolean-algebra-access-control.ipynb`
